# Artificial Neural Network (ANN) — Car Purchase Amount Prediction

## Overview

This notebook demonstrates a **feedforward Artificial Neural Network (ANN)** built with **PyTorch** to predict the amount a customer will spend on a car based on demographic and financial features.

### What is an ANN?

An Artificial Neural Network is a computational model inspired by biological neural networks. It consists of layers of interconnected nodes ("neurons"):

- **Input Layer** — receives the raw features (one neuron per feature)
- **Hidden Layers** — learn increasingly abstract representations through weighted connections, activation functions (ReLU), and dropout regularisation
- **Output Layer** — produces the final prediction (one neuron for regression)

During training, the network minimises a loss function (MSE for regression) using **backpropagation** and **gradient descent**, iteratively adjusting weights to reduce prediction error.

### Dataset

The Car Purchasing dataset contains **500 samples** with:
- **Input features (5):** gender, age, annual salary, credit card debt, net worth
- **Target:** car purchase amount (continuous, in dollars)

### Notebook Outline
1. Setup & Imports
2. Data Exploration
3. Data Preprocessing
4. Model Architecture
5. Training
6. Evaluation
7. Single Prediction
8. Save Model

---
## 1. Setup & Imports

In [ ]:
import os
import sys
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")

sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

from data_loader import (
    load_car_data,
    split_features_target,
    get_scalers,
    transform_data,
    create_tensor_dataset,
    create_dataloaders,
    FEATURE_COLUMNS,
    TARGET_COLUMN,
)
from model import (
    CarPurchaseANN,
    train_model,
    predict,
    predict_single,
    compute_metrics,
    save_model,
)
from visualization import (
    plot_target_distribution,
    plot_feature_distributions,
    plot_correlation_heatmap,
    plot_feature_vs_target,
    plot_training_history,
    plot_lr_schedule,
    plot_predictions_vs_actual,
    plot_residuals,
    plot_error_metrics,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}  |  Device: {device}")

---
## 2. Data Exploration

In [ ]:
df = load_car_data()

print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFeatures : {FEATURE_COLUMNS}")
print(f"Target   : {TARGET_COLUMN}")
df.head()

In [ ]:
df.describe()

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

In [ ]:
plot_target_distribution(df[TARGET_COLUMN])
plt.show()

In [ ]:
fig = plot_feature_distributions(df, FEATURE_COLUMNS)
plt.show()

In [ ]:
plot_correlation_heatmap(df)
plt.show()

In [ ]:
fig = plot_feature_vs_target(df, FEATURE_COLUMNS, TARGET_COLUMN)
plt.show()

---
## 3. Data Preprocessing

We split the data into **train (70%)**, **validation (15%)**, and **test (15%)** sets, then apply **MinMaxScaler** to normalise both features and target to [0, 1]. Scaling is essential for neural networks to converge efficiently.

In [ ]:
from sklearn.model_selection import train_test_split

X, y = split_features_target(df)

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=RANDOM_STATE,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.15 / 0.85, random_state=RANDOM_STATE,
)

print(f"Train: {len(X_train):,}  |  Val: {len(X_val):,}  |  Test: {len(X_test):,}")

In [ ]:
X_train_sc, y_train_sc, feature_scaler, target_scaler = get_scalers(X_train, y_train)
X_val_sc, y_val_sc = transform_data(X_val, y_val, feature_scaler, target_scaler)
X_test_sc, y_test_sc = transform_data(X_test, y_test, feature_scaler, target_scaler)

print(f"Scaled feature range: [{X_train_sc.min():.4f}, {X_train_sc.max():.4f}]")
print(f"Scaled target  range: [{y_train_sc.min():.4f}, {y_train_sc.max():.4f}]")

In [ ]:
train_ds = create_tensor_dataset(X_train_sc, y_train_sc)
val_ds = create_tensor_dataset(X_val_sc, y_val_sc)
test_ds = create_tensor_dataset(X_test_sc, y_test_sc)

BATCH_SIZE = 32
train_loader, val_loader, test_loader = create_dataloaders(
    train_ds, val_ds, test_ds, batch_size=BATCH_SIZE,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches  : {len(val_loader)}")
print(f"Test batches : {len(test_loader)}")

---
## 4. Model Architecture

Our ANN has:
- **Input layer**: 5 neurons (one per feature)
- **Hidden layer 1**: 64 neurons + ReLU + Dropout(0.2)
- **Hidden layer 2**: 32 neurons + ReLU + Dropout(0.2)
- **Output layer**: 1 neuron (regression target)

Total architecture: `5 → 64 → 32 → 1`

In [ ]:
model = CarPurchaseANN(input_dim=5, hidden_dims=(64, 32), dropout=0.2)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"\nModel architecture:\n{model}")

---
## 5. Training

We use:
- **Loss**: MSELoss (Mean Squared Error — standard for regression)
- **Optimizer**: Adam with learning rate 1e-3
- **Scheduler**: ReduceLROnPlateau — automatically reduces LR when validation loss plateaus

In [ ]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=10, verbose=False,
)

NUM_EPOCHS = 100

print(f"Training for {NUM_EPOCHS} epochs...\n")
model, history = train_model(
    model, train_loader, val_loader, criterion,
    optimizer, scheduler,
    device=device, num_epochs=NUM_EPOCHS,
)

In [ ]:
fig = plot_training_history(history)
plt.show()

In [ ]:
fig = plot_lr_schedule(history)
plt.show()

---
## 6. Evaluation

We evaluate the trained model on the **held-out test set**, inverse-transforming predictions back to the original dollar scale.

In [ ]:
preds_scaled, targets_scaled = predict(model, test_loader, device)

y_pred_original = target_scaler.inverse_transform(preds_scaled.reshape(-1, 1)).flatten()
y_true_original = target_scaler.inverse_transform(targets_scaled.reshape(-1, 1)).flatten()

metrics = compute_metrics(y_true_original, y_pred_original)

print("Test Set Metrics (original scale):")
print(f"  MSE  : {metrics['mse']:>14,.2f}")
print(f"  RMSE : {metrics['rmse']:>14,.2f}")
print(f"  MAE  : {metrics['mae']:>14,.2f}")
print(f"  R²   : {metrics['r2']:>14.4f}")

In [ ]:
plot_predictions_vs_actual(y_true_original, y_pred_original)
plt.show()

In [ ]:
fig = plot_residuals(y_true_original, y_pred_original)
plt.show()

In [ ]:
plot_error_metrics(metrics)
plt.show()

---
## 7. Single Prediction

Demonstrate predicting the car purchase amount for an individual customer.

In [ ]:
sample_idx = 0
sample_row = X_test.iloc[sample_idx]
sample_true = y_test.iloc[sample_idx]

sample_scaled = feature_scaler.transform(sample_row.values.reshape(1, -1))
sample_tensor = torch.tensor(sample_scaled, dtype=torch.float32).squeeze(0)

pred_scaled = predict_single(model, sample_tensor, device)
pred_original = target_scaler.inverse_transform([[pred_scaled]])[0, 0]

print("Sample Customer:")
for feat in FEATURE_COLUMNS:
    print(f"  {feat:>20s}: {sample_row[feat]:>12,.2f}")
print(f"\n  {'Actual Purchase':>20s}: ${sample_true:>12,.2f}")
print(f"  {'Predicted Purchase':>20s}: ${pred_original:>12,.2f}")
print(f"  {'Error':>20s}: ${abs(sample_true - pred_original):>12,.2f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

colors = sns.color_palette("Set2", len(FEATURE_COLUMNS))
ax1.barh(FEATURE_COLUMNS, sample_row.values, color=colors)
ax1.set_title("Customer Features", fontsize=12, fontweight="bold")
ax1.set_xlabel("Value")

bar_colors = ["#66c2a5", "#fc8d62"]
bars = ax2.bar(["Actual", "Predicted"], [sample_true, pred_original], color=bar_colors)
for bar, val in zip(bars, [sample_true, pred_original]):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
             f"${val:,.0f}", ha="center", va="bottom", fontweight="bold")
ax2.set_title("Actual vs Predicted", fontsize=12, fontweight="bold")
ax2.set_ylabel("Car Purchase Amount ($)")

plt.tight_layout()
plt.show()

---
## 8. Save Model

In [ ]:
model_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "ann_car_purchase.pth")
save_model(model, model_path)
print(f"Model saved to: {model_path}")

file_size_kb = os.path.getsize(model_path) / 1024
print(f"Model file size: {file_size_kb:.1f} KB")

---
## Summary

| Aspect | Details |
|--------|--------|
| Task | Regression — predict car purchase amount |
| Architecture | Feedforward ANN: 5 → 64 → 32 → 1 |
| Activation | ReLU (hidden layers) |
| Regularisation | Dropout (0.2), weight decay (1e-5) |
| Loss | MSE (Mean Squared Error) |
| Optimizer | Adam (lr=1e-3) |
| Scheduler | ReduceLROnPlateau (factor=0.5, patience=10) |
| Epochs | 100 |
| Data Scaling | MinMaxScaler on features and target |

**Key Takeaways:**
- Even a simple 2-hidden-layer ANN can capture non-linear relationships between customer demographics/finances and purchase behaviour
- MinMaxScaling is critical for stable and fast convergence
- The R² score shows how well the model explains variance in the data
- Residual analysis helps detect any systematic bias in predictions
- ReduceLROnPlateau automatically adapts the learning rate, preventing overshooting during later training stages